# 01. Data ingestion and cleaning

This notebook creates canonical in-memory analysis tables from the AI-vs-AI Among Us experiment corpus. It is read-only: it never modifies files in `data/` or writes derived outputs.

The unit of analysis is a game, meeting utterance, or impostor deception record. The final game summaries are newline-delimited JSON (one game per line), while meeting and deception annotations are CSV files.

## Reproducibility

Run from the repository root after installing development dependencies:

```bash
make install-dev
venv/bin/jupyter lab
```

Expected raw-file pattern: `data/<provider>/<condition>_{discussion,deceitLog,summary}.{csv,json}`. Files such as `.DS_Store` and `:Zone.Identifier` are ignored.

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

In [2]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml and data/.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {DATA_DIR}")

Project root: /home/tweninge/projects/amongus
Raw data: /home/tweninge/projects/amongus/data


## Discover raw files

Conditions use the form `CVK`: `C` crewmates and `K` impostors. Total players are derived as `C + K`. The filename casing is normalized during ingestion.

In [3]:
CONDITION_PATTERN = re.compile(r"^(?P<crewmates>\d+)[Vv](?P<impostors>\d+)_")


def parse_file_metadata(path: Path) -> dict[str, object]:
    match = CONDITION_PATTERN.match(path.name)
    if not match:
        raise ValueError(f"Unrecognized experiment filename: {path}")
    return {
        "provider": path.parent.name.upper(),
        "condition": f"{match['crewmates']}V{match['impostors']}",
        "num_crewmates": int(match['crewmates']),
        "num_impostors": int(match['impostors']),
        "num_players": int(match['crewmates']) + int(match['impostors']),
        "impostor_ratio": int(match['impostors']) / (int(match['crewmates']) + int(match['impostors'])),
        "source_file": str(path.relative_to(PROJECT_ROOT)),
    }


def experiment_files(suffix: str) -> list[Path]:
    return sorted(
        path
        for path in DATA_DIR.glob(f"*/*{suffix}")
        if "Zone.Identifier" not in path.name and path.name != ".DS_Store"
    )


discussion_files = experiment_files("_discussion.csv")
deception_files = [
    path
    for path in DATA_DIR.glob("*/*_deceitLog.csv")
    if "Zone.Identifier" not in path.name
]
summary_files = experiment_files("_summary.json")

file_inventory = pd.DataFrame(
    [
        {**parse_file_metadata(path), "file_type": "discussion"}
        for path in discussion_files
    ]
    + [
        {**parse_file_metadata(path), "file_type": "deception"}
        for path in deception_files
    ]
    + [
        {**parse_file_metadata(path), "file_type": "summary"}
        for path in summary_files
    ]
)

display(file_inventory.pivot_table(index=["provider", "condition"], columns="file_type", values="source_file", aggfunc="size", fill_value=0))

file_type           deception  discussion  summary
provider condition                                
CLAUDE   3V1                1           1        1
         4V1                1           1        1
         4V2                1           1        1
         5V1                1           1        1
         5V2                1           1        1
         5V3                1           1        1
         6V1                1           1        1
         6V2                1           1        1
         6V3                1           1        1
         7V1                1           1        1
         7V2                1           1        1
GEMINI   3V1                1           1        1
         4V1                1           1        1
         4V2                1           1        1
         5V1                1           1        1
         5V2                1           1        1
         5V3                1           1        1
         6V1                1           1        1
         6V2                1           1        1
         6V3                1           1        1
         7V1                1           1        1
         7V2                1           1        1
LLAMA    3V1                1           1        1
         4V1                1           1        1
         4V2                1           1        1
         4V3                1           1        0
         5V1                1           1        1
         5V2                1           1        1
         5V3                1           1        1
         6V1                1           1        1
         6V2                1           1        1
         6V3                1           1        1
         7V1                1           1        1
         7V2                1           1        1
OPENAI   3V1                1           1        1
         4V1                1           1        1
         4V2                1           1        1
         5V1                1           1        1
         5V2                1           1        1
         5V3                1           1        1
         6V1                1           1        1
         6V2                1           1        1
         6V3                1           1        1
         7V1                1           1        1
         7V2                1           1        1

## Shared cleaning helpers

Raw labels are retained alongside normalized labels. Missing annotations remain missing values rather than being treated as a communication category.

In [4]:
SPEECH_ACT_MAP = {
    "directive": "Directive",
    "directives": "Directive",
    "representative": "Representative",
    "representatives": "Representative",
    "commissive": "Commissive",
    "commissives": "Commissive",
    "expressive": "Expressive",
    "declaration": "Declaration",
}
DECEPTION_MAP = {
    "concealment": "Concealment",
    "equivocation": "Equivocation",
    "falsification": "Falsification",
}
WINNER_SIDE = {1: "Impostor", 2: "Crewmate", 3: "Crewmate", 4: "Impostor"}
TERMINATION_TYPE = {1: "parity", 2: "ejection", 3: "tasks", 4: "time_limit"}


def normalize_label(value: object, mapping: dict[str, str]) -> object:
    if pd.isna(value):
        return pd.NA
    cleaned = str(value).strip()
    if not cleaned or cleaned.upper() == "MISSING":
        return pd.NA
    return mapping.get(cleaned.lower(), pd.NA)


def game_number(value: object) -> object:
    match = re.search(r"(\d+)", str(value))
    return int(match.group(1)) if match else pd.NA


def game_key(frame: pd.DataFrame) -> pd.Series:
    return frame["provider"] + "|" + frame["condition"] + "|" + frame["game_number"].astype("string")

## Ingest meeting discussion

Discussion files contain the meeting utterance, game timestep, speaker role, and speech-act annotation. Individual speaker IDs are not available in this export.

In [5]:
def load_discussion(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    metadata = parse_file_metadata(path)
    frame = frame.rename(columns={"game": "game_id", "action": "utterance", "category1": "speech_act_raw"})
    frame = frame.assign(**metadata)
    frame["game_number"] = frame["game_id"].map(game_number).astype("Int64")
    frame["timestep"] = pd.to_numeric(frame["timestep"], errors="coerce").astype("Int64")
    frame["player_identity"] = frame["player_identity"].astype("string").str.strip()
    frame["speech_act"] = frame["speech_act_raw"].map(lambda value: normalize_label(value, SPEECH_ACT_MAP))
    frame["label_status"] = frame["speech_act"].notna().map({True: "annotated", False: "missing_or_unrecognized"})
    return frame


discussion = pd.concat([load_discussion(path) for path in discussion_files], ignore_index=True)
discussion["game_key"] = game_key(discussion)
discussion["utterance_order"] = discussion.groupby("game_key").cumcount() + 1

display(discussion.head())
display(discussion.groupby(["provider", "speech_act"], dropna=False).size().rename("utterances").reset_index())

,game_id,timestep,player_identity,utterance,speech_act_raw,provider,condition,num_crewmates,num_impostors,num_players,impostor_ratio,source_file,game_number,speech_act,label_status,game_key,utterance_order
0,Game 1,14,Crewmate,I found the dead body at Communications. I was...,Representative,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Representative,annotated,CLAUDE|3V1|1,1
1,Game 1,14,Crewmate,I need to defend myself here. My actual path w...,Representative,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Representative,annotated,CLAUDE|3V1|1,2
2,Game 1,14,Impostor,"Thanks for asking, blue. My path was: I comple...",Directive,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Directive,annotated,CLAUDE|3V1|1,3
3,Game 1,14,Crewmate,Let me be very clear about why blue is suspici...,Directive,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Directive,annotated,CLAUDE|3V1|1,4
4,Game 1,14,Crewmate,"Black, your story has a critical flaw. You cla...",Directive,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Directive,annotated,CLAUDE|3V1|1,5


,provider,speech_act,utterances
0,CLAUDE,Commissive,3259
1,CLAUDE,Declaration,49
2,CLAUDE,Directive,9230
3,CLAUDE,Expressive,379
4,CLAUDE,Representative,6103
5,CLAUDE,NaN,30
6,GEMINI,Commissive,324
7,GEMINI,Declaration,3
8,GEMINI,Directive,17886
9,GEMINI,Expressive,274


## Ingest impostor deception records

In [6]:
def load_deception(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    metadata = parse_file_metadata(path)
    frame = frame.rename(columns={"game_index": "game_id", "text": "deception_text", "category1": "deception_raw"})
    frame = frame.assign(**metadata)
    frame["game_number"] = frame["game_id"].map(game_number).astype("Int64")
    frame["deception_type"] = frame["deception_raw"].map(lambda value: normalize_label(value, DECEPTION_MAP))
    frame["label_status"] = frame["deception_type"].notna().map({True: "annotated", False: "missing_or_unrecognized"})
    return frame


deception = pd.concat([load_deception(path) for path in deception_files], ignore_index=True)
deception["game_key"] = game_key(deception)
deception["record_order"] = deception.groupby("game_key").cumcount() + 1

display(deception.head())
display(deception.groupby(["provider", "deception_type"], dropna=False).size().rename("records").reset_index())

,game_id,imposter,discussion,deception_text,deception_raw,provider,condition,num_crewmates,num_impostors,num_players,impostor_ratio,source_file,game_number,deception_type,label_status,game_key,record_order
0,Game 49,Player 2: black,NaN,"Hey everyone, who called the emergency meeting...",Concealment,CLAUDE,5V3,5,3,8,0.375,data/CLAUDE/5V3_deceitLog.csv,49,Concealment,annotated,CLAUDE|5V3|49,1
1,Game 49,Player 2: black,"Player 3: orange: Hey everyone, I'm curious wh...","Thanks for the info, red - I'm sorry to hear a...",Falsification,CLAUDE,5V3,5,3,8,0.375,data/CLAUDE/5V3_deceitLog.csv,49,Falsification,annotated,CLAUDE|5V3|49,2
2,Game 49,Player 2: black,"Player 3: orange: I did speak in round 0, blac...","Okay, I hear everyone loud and clear - let me ...",Equivocation,CLAUDE,5V3,5,3,8,0.375,data/CLAUDE/5V3_deceitLog.csv,49,Equivocation,annotated,CLAUDE|5V3|49,3
3,Game 49,Player 5: green,"Player 2: black: Hey everyone, who called the ...","Hey everyone, I'm Player 5 green. I was also i...",Equivocation,CLAUDE,5V3,5,3,8,0.375,data/CLAUDE/5V3_deceitLog.csv,49,Equivocation,annotated,CLAUDE|5V3|49,4
4,Game 49,Player 5: green,"Player 6: purple: Hey everyone, I'm Player 6 p...",I want to address the questions being raised a...,Falsification,CLAUDE,5V3,5,3,8,0.375,data/CLAUDE/5V3_deceitLog.csv,49,Falsification,annotated,CLAUDE|5V3|49,5


,provider,deception_type,records
0,CLAUDE,Concealment,1007
1,CLAUDE,Equivocation,1830
2,CLAUDE,Falsification,2049
3,CLAUDE,NaN,28
4,GEMINI,Concealment,1670
5,GEMINI,Equivocation,2972
6,GEMINI,Falsification,1541
7,GEMINI,NaN,63
8,LLAMA,Concealment,1774
9,LLAMA,Equivocation,5559


## Ingest final game summaries

Each summary file is JSON Lines: every line is a one-key object whose key is the game ID and whose value contains configuration, player metadata, and the final outcome.

In [7]:
def load_summary(path: Path) -> pd.DataFrame:
    metadata = parse_file_metadata(path)
    records = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if len(record) != 1:
                raise ValueError(f"Expected one game per line in {path}, line {line_number}.")
            game_id, payload = next(iter(record.items()))
            config = payload.get("config", {})
            player_records = [value for key, value in payload.items() if key.startswith("Player ")]
            models = sorted({player.get("model") for player in player_records if player.get("model")})
            records.append(
                {
                    **metadata,
                    "game_id": game_id,
                    "game_number": game_number(game_id),
                    "winner_code": payload.get("winner"),
                    "winner_reason": payload.get("winner_reason"),
                    "winner_side": WINNER_SIDE.get(payload.get("winner"), pd.NA),
                    "termination_type": TERMINATION_TYPE.get(payload.get("winner"), pd.NA),
                    "model": " | ".join(models),
                    "player_models_unique": len(models),
                    "config_num_players": config.get("num_players"),
                    "config_num_impostors": config.get("num_impostors"),
                    "max_timesteps": config.get("max_timesteps"),
                    "discussion_rounds": config.get("discussion_rounds"),
                }
            )
    return pd.DataFrame(records)


games = pd.concat([load_summary(path) for path in summary_files], ignore_index=True)
games["game_number"] = games["game_number"].astype("Int64")
games["winner_code"] = games["winner_code"].astype("Int64")
games["game_key"] = game_key(games)
games["impostor_win"] = games["winner_side"].eq("Impostor")

assert games["game_key"].is_unique, "A game has more than one final summary record."
assert games["winner_side"].notna().all(), "An outcome code is not mapped to a winning side."
assert games["num_players"].eq(games["config_num_players"]).all(), "Filename condition and summary player count disagree."
assert games["num_impostors"].eq(games["config_num_impostors"]).all(), "Filename condition and summary impostor count disagree."

display(games.head())
display(games.groupby("provider")["impostor_win"].agg(games="size", impostor_win_rate="mean"))

,provider,condition,num_crewmates,num_impostors,num_players,impostor_ratio,source_file,game_id,game_number,winner_code,winner_reason,winner_side,termination_type,model,player_models_unique,config_num_players,config_num_impostors,max_timesteps,discussion_rounds,game_key,impostor_win
0,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 36,36,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|36,True
1,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 12,12,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|12,True
2,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 40,40,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|40,True
3,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 19,19,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|19,True
4,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 2,2,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|2,True


,games,impostor_win_rate
provider,,
CLAUDE,1100,0.462727
GEMINI,1100,0.527273
LLAMA,1100,0.755455
OPENAI,1100,0.550909


## Coverage and data-quality audit

A game can have a final outcome without a meeting transcript. This audit identifies summary games with no discussion or deception record, plus any transcript that lacks a matching final summary.

In [8]:
def unique_game_table(frame: pd.DataFrame, name: str) -> pd.DataFrame:
    return frame[["provider", "condition", "game_key"]].drop_duplicates().assign(**{name: True})


coverage = games[["provider", "condition", "game_key"]].copy()
coverage = coverage.merge(unique_game_table(discussion, "has_discussion"), how="left", on=["provider", "condition", "game_key"])
coverage = coverage.merge(unique_game_table(deception, "has_deception"), how="left", on=["provider", "condition", "game_key"])
coverage[["has_discussion", "has_deception"]] = coverage[["has_discussion", "has_deception"]].fillna(False)

discussion_without_summary = discussion.loc[~discussion["game_key"].isin(games["game_key"])]
deception_without_summary = deception.loc[~deception["game_key"].isin(games["game_key"])]

coverage_summary = coverage.groupby(["provider", "condition"]).agg(
    games=("game_key", "size"),
    games_with_discussion=("has_discussion", "sum"),
    games_with_deception=("has_deception", "sum"),
).reset_index()
coverage_summary["discussion_coverage"] = coverage_summary["games_with_discussion"] / coverage_summary["games"]
coverage_summary["deception_coverage"] = coverage_summary["games_with_deception"] / coverage_summary["games"]

print(f"Discussion rows without a final outcome: {len(discussion_without_summary):,}")
print(f"Deception rows without a final outcome: {len(deception_without_summary):,}")
display(coverage_summary)

label_audit = pd.concat(
    [
        discussion.assign(dataset="discussion", normalized_label=discussion["speech_act"])[["dataset", "provider", "condition", "speech_act_raw", "normalized_label"]].rename(columns={"speech_act_raw": "raw_label"}),
        deception.assign(dataset="deception", normalized_label=deception["deception_type"])[["dataset", "provider", "condition", "deception_raw", "normalized_label"]].rename(columns={"deception_raw": "raw_label"}),
    ],
    ignore_index=True,
)
display(label_audit.groupby(["dataset", "provider", "raw_label", "normalized_label"], dropna=False).size().rename("rows").reset_index())

Discussion rows without a final outcome: 1,841
Deception rows without a final outcome: 597


,provider,condition,games,games_with_discussion,games_with_deception,discussion_coverage,deception_coverage
0,CLAUDE,3V1,100,57,57,0.57,0.57
1,CLAUDE,4V1,100,79,79,0.79,0.79
2,CLAUDE,4V2,100,39,39,0.39,0.39
3,CLAUDE,5V1,100,86,86,0.86,0.86
4,CLAUDE,5V2,100,84,84,0.84,0.84
5,CLAUDE,5V3,100,31,31,0.31,0.31
6,CLAUDE,6V1,100,83,83,0.83,0.83
7,CLAUDE,6V2,100,96,96,0.96,0.96
8,CLAUDE,6V3,100,74,74,0.74,0.74
9,CLAUDE,7V1,100,87,87,0.87,0.87


,dataset,provider,raw_label,normalized_label,rows
0,deception,CLAUDE,Concealment,Concealment,1007
1,deception,CLAUDE,Equivocation,Equivocation,1830
2,deception,CLAUDE,Falsification,Falsification,2049
3,deception,CLAUDE,MISSING,NaN,28
4,deception,GEMINI,Concealment,Concealment,1670
5,deception,GEMINI,Equivocation,Equivocation,2972
6,deception,GEMINI,Falsification,Falsification,1541
7,deception,GEMINI,MISSING,NaN,63
8,deception,LLAMA,Concealment,Concealment,1774
9,deception,LLAMA,Equivocation,Equivocation,5559


## Preview canonical analysis tables

These in-memory tables are the intended inputs for later descriptive and modeling notebooks. For now, this notebook previews them only; it does not write files.

In [9]:
tables = {
    "games": games,
    "discussion": discussion,
    "deception": deception,
    "coverage": coverage,
    "coverage_summary": coverage_summary,
    "label_audit": label_audit,
}

for name, frame in tables.items():
    print(f"\n{name}: {len(frame):,} rows, {len(frame.columns)} columns")
    display(frame.head(3))


games: 4,400 rows, 21 columns


,provider,condition,num_crewmates,num_impostors,num_players,impostor_ratio,source_file,game_id,game_number,winner_code,winner_reason,winner_side,termination_type,model,player_models_unique,config_num_players,config_num_impostors,max_timesteps,discussion_rounds,game_key,impostor_win
0,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 36,36,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|36,True
1,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 12,12,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|12,True
2,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_summary.json,Game 40,40,1,Impostors win! (Crewmates being outnumbered or...,Impostor,parity,claude-sonnet-4-6,1,4,1,20,3,CLAUDE|3V1|40,True



discussion: 120,460 rows, 17 columns


,game_id,timestep,player_identity,utterance,speech_act_raw,provider,condition,num_crewmates,num_impostors,num_players,impostor_ratio,source_file,game_number,speech_act,label_status,game_key,utterance_order
0,Game 1,14,Crewmate,I found the dead body at Communications. I was...,Representative,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Representative,annotated,CLAUDE|3V1|1,1
1,Game 1,14,Crewmate,I need to defend myself here. My actual path w...,Representative,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Representative,annotated,CLAUDE|3V1|1,2
2,Game 1,14,Impostor,"Thanks for asking, blue. My path was: I comple...",Directive,CLAUDE,3V1,3,1,4,0.25,data/CLAUDE/3V1_discussion.csv,1,Directive,annotated,CLAUDE|3V1|1,3



deception: 27,297 rows, 17 columns


...

,provider,condition,game_key,has_discussion,has_deception
0,CLAUDE,3V1,CLAUDE|3V1|36,False,False
1,CLAUDE,3V1,CLAUDE|3V1|12,False,False
2,CLAUDE,3V1,CLAUDE|3V1|40,False,False



coverage_summary: 44 rows, 7 columns


,provider,condition,games,games_with_discussion,games_with_deception,discussion_coverage,deception_coverage
0,CLAUDE,3V1,100,57,57,0.57,0.57
1,CLAUDE,4V1,100,79,79,0.79,0.79
2,CLAUDE,4V2,100,39,39,0.39,0.39



label_audit: 147,757 rows, 5 columns


,dataset,provider,condition,raw_label,normalized_label
0,discussion,CLAUDE,3V1,Representative,Representative
1,discussion,CLAUDE,3V1,Representative,Representative
2,discussion,CLAUDE,3V1,Directive,Directive


## Raw compact-log event source

The raw archive is large, so this notebook streams `agent-logs-compact.json` one JSON object at a time and never loads an entire log into memory. It is opt-in and read-only: leave `RAW_LOG_MODE` set to `"off"` during ordinary notebook work, use `"sample"` to validate the parser on one file, and use `"all"` only when building the full in-memory event table.

Each record describes a model decision and the state visible to that model before the decision. A decision is retained as an `agent_decision` event; confirmed kills, reports, votes, and ejections will be reconstructed separately from successive state snapshots rather than assumed from a requested action.

In [10]:
RAW_LOG_MODE = "all"  # "off", "sample", or "all"
RAW_LOG_DIR = DATA_DIR / "logs"
RAW_CONDITION_PATTERN = re.compile(r"^(?P<provider>[A-Z]+)_(?P<crewmates>\d+)[Vv](?P<impostors>\d+)$")


def parse_raw_log_metadata(path: Path) -> dict[str, object]:
    match = RAW_CONDITION_PATTERN.match(path.parent.name)
    if not match:
        raise ValueError(f"Unrecognized compact-log path: {path}")
    crewmates, impostors = int(match["crewmates"]), int(match["impostors"])
    return {
        "provider": match["provider"],
        "condition": f"{crewmates}V{impostors}",
        "num_crewmates": crewmates,
        "num_impostors": impostors,
        "num_players": crewmates + impostors,
        "impostor_ratio": impostors / (crewmates + impostors),
        "raw_log_file": str(path.relative_to(PROJECT_ROOT)),
    }


def compact_log_files() -> list[Path]:
    return sorted(RAW_LOG_DIR.glob("*/*/agent-logs-compact.json"))


def stream_concatenated_json(path: Path, chunk_size: int = 1 << 20):
    """Yield adjacent JSON objects without loading a complete log file."""
    decoder, buffer, sequence = json.JSONDecoder(), "", 0
    with path.open(encoding="utf-8") as handle:
        while chunk := handle.read(chunk_size):
            buffer += chunk
            while True:
                buffer = buffer.lstrip()
                if not buffer:
                    break
                try:
                    record, end = decoder.raw_decode(buffer)
                except json.JSONDecodeError:
                    break
                sequence += 1
                yield sequence, record
                buffer = buffer[end:]
    if buffer.strip():
        raise ValueError(f"Incomplete JSON record at the end of {path}")


def color_from_player_label(value: object) -> object:
    match = re.match(r"^Player\s+\d+\s*:\s*(?P<color>.+?)\s*$", str(value))
    if not match:
        return pd.NA
    return re.sub(r"\s+\([^)]*\)$", "", match["color"]).strip().lower()


def action_from_interaction(interaction: object) -> tuple[object, str]:
    if not isinstance(interaction, dict):
        return pd.NA, "missing_interaction"
    response = interaction.get("response")
    if isinstance(response, dict):
        if isinstance(response.get("Action"), str):
            return response["Action"].strip(), "response.action"
        thinking = response.get("Thinking Process")
        if isinstance(thinking, dict) and isinstance(thinking.get("action"), str):
            return thinking["action"].strip(), "response.thinking_process.action"
    if isinstance(response, str) and response.strip():
        return response.strip(), "response.string"
    full_response = interaction.get("full_response")
    response_text, response_source = pd.NA, "missing_action"
    if isinstance(full_response, str):
        response_text, response_source = full_response, "full_response.string"
    elif isinstance(full_response, dict):
        message = full_response.get("message")
        if isinstance(message, dict) and isinstance(message.get("content"), str):
            response_text, response_source = message["content"], "full_response.message.content"
        elif isinstance(message, str):
            response_text, response_source = message, "full_response.message"
        else:
            choices = full_response.get("choices")
            if isinstance(choices, list) and choices:
                choice_message = choices[0].get("message", {}) if isinstance(choices[0], dict) else {}
                if isinstance(choice_message, dict) and isinstance(choice_message.get("content"), str):
                    response_text, response_source = choice_message["content"], "full_response.choices[0].message.content"
    if isinstance(response_text, str):
        match = re.search(r"\[Action\]\s*(?P<action>.+)", response_text, flags=re.IGNORECASE | re.DOTALL)
        if match:
            action = match["action"].strip()
            if action.upper().startswith("SPEAK:"):
                return action, f"{response_source}.action_tag"
            return action.splitlines()[0], f"{response_source}.action_tag"
    return pd.NA, "missing_action"


def normalize_logged_action(action: object) -> object:
    if pd.isna(action):
        return pd.NA
    return re.sub(r"^\s*(?:\d+\s*[.:]\s*)+", "", str(action)).strip()


def action_type(action: object) -> object:
    if pd.isna(action):
        return pd.NA
    normalized = str(normalize_logged_action(action)).upper()
    for prefix, event_type in {
        "MOVE": "move", "KILL": "kill_attempt", "COMPLETE TASK": "task_attempt",
        "COMPLETE FAKE TASK": "task_attempt", "VIEW MONITOR": "monitor_view",
        "SPEAK": "speech", "VOTE": "vote_attempt", "REPORT": "report_attempt",
        "CALL MEETING": "meeting_call_attempt", "VENT": "vent_attempt",
    }.items():
        if normalized.startswith(prefix):
            return event_type
    if normalized.startswith("I WILL MOVE FROM"):
        return "move"
    return "unparsed_action"


def parse_visible_state(all_info: object) -> dict[str, object]:
    text = all_info if isinstance(all_info, str) else ""
    time_match = re.search(r"^Game Time:\s*(?P<time>\d+)(?:/(?P<maximum>\d+))?", text, flags=re.MULTILINE)
    location_match = re.search(r"^Current Location:\s*(?P<location>.+)$", text, flags=re.MULTILINE)
    roster_match = re.search(r"^Players in (?P<room>[^:]+):\s*(?P<players>.+)$", text, flags=re.MULTILINE)
    phase_match = re.search(r"^Current phase:\s*(?P<phase>.+)$", text, flags=re.MULTILINE)
    meeting_round_match = re.search(r"Discussion round \((?P<round>\d+)/\d+\)", text)
    actions_match = re.search(r"Available actions:\s*(?P<actions>.*)$", text, flags=re.MULTILINE | re.DOTALL)
    visible_players, visible_locations, visible_dead_players = [], {}, []
    if roster_match:
        room = roster_match["room"].strip()
        visible_players = re.findall(r"Player\s+\d+\s*:\s*[^,]+", roster_match["players"])
        visible_locations = {color_from_player_label(player): room for player in visible_players}
        visible_dead_players = [color_from_player_label(player) for player in visible_players if re.search(r"\(dead\)\s*$", player, flags=re.IGNORECASE)]
    ejection_match = re.search(r"MEETING RESULT:\s*(?P<player>Player\s+\d+\s*:\s*[^.]+?)\s+was ejected", text, flags=re.IGNORECASE)
    available_actions = []
    if actions_match:
        available_actions = [re.sub(r"^\d+\.\s*", "", line).strip() for line in actions_match["actions"].splitlines() if re.match(r"^\d+\.\s+", line.strip())]
    return {
        "observed_timestep": int(time_match["time"]) if time_match else pd.NA,
        "max_timesteps_observed": int(time_match["maximum"]) if time_match and time_match["maximum"] else pd.NA,
        "phase_raw": phase_match["phase"].strip() if phase_match else pd.NA,
        "meeting_round": int(meeting_round_match["round"]) if meeting_round_match else pd.NA,
        "location": location_match["location"].strip() if location_match else pd.NA,
        "visible_players": json.dumps(visible_players),
        "visible_player_locations": json.dumps(visible_locations),
        "visible_dead_players": json.dumps(visible_dead_players),
        "observed_ejection_player": color_from_player_label(ejection_match["player"]) if ejection_match else pd.NA,
        "available_actions": json.dumps(available_actions),
    }


def load_raw_decision_events(mode: str = RAW_LOG_MODE) -> pd.DataFrame:
    if mode not in {"off", "sample", "all"}:
        raise ValueError("RAW_LOG_MODE must be 'off', 'sample', or 'all'.")
    files = compact_log_files()
    if mode == "off":
        return pd.DataFrame()
    if mode == "sample":
        files = files[:1]
    rows = []
    for path in files:
        metadata = parse_raw_log_metadata(path)
        for sequence, record in stream_concatenated_json(path):
            interaction, player = record.get("interaction", {}), record.get("player", {})
            prompt = interaction.get("prompt", {}) if isinstance(interaction, dict) else {}
            action, action_source = action_from_interaction(interaction)
            action = normalize_logged_action(action)
            state = parse_visible_state(prompt.get("All Info") if isinstance(prompt, dict) else pd.NA)
            game_id, game_number_value = record.get("game_index"), game_number(record.get("game_index"))
            rows.append({
                **metadata, "game_id": game_id, "game_number": game_number_value,
                "game_key": f"{metadata['provider']}|{metadata['condition']}|{game_number_value}",
                "log_sequence": sequence, "timestep": record.get("step"), "timestamp": record.get("timestamp"),
                "actor_id": color_from_player_label(player.get("name")), "actor_role": player.get("identity"),
                "model": player.get("model"), "action_text": action, "action_source": action_source,
                "event_type": action_type(action),
                "phase": "meeting" if str(state["phase_raw"]).startswith("Meeting") else "task", **state,
            })
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    frame["timestep"] = pd.to_numeric(frame["timestep"], errors="coerce").astype("Int64")
    instance_groups = ["raw_log_file", "game_key"]
    frame["raw_game_instance"] = (
        frame.groupby(instance_groups, sort=False)["timestep"].diff().lt(0).fillna(False)
        .groupby([frame[column] for column in instance_groups], sort=False).cumsum().add(1).astype("Int64")
    )
    frame["raw_game_instance_key"] = frame["game_key"] + "|raw_instance_" + frame["raw_game_instance"].astype("string")
    roster = (
        frame.groupby(["raw_game_instance_key", "actor_role"], dropna=False)["actor_id"]
        .nunique().unstack(fill_value=0).rename(columns={"Crewmate": "observed_crewmates", "Impostor": "observed_impostors"})
        .reset_index()
    )
    for column in ["observed_crewmates", "observed_impostors"]:
        if column not in roster:
            roster[column] = 0
    instance_metadata = frame.groupby("raw_game_instance_key", as_index=False)[["game_key", "num_crewmates", "num_impostors"]].first().merge(roster, on="raw_game_instance_key", how="left")
    instance_metadata["roster_is_compatible"] = (
        instance_metadata["observed_crewmates"].le(instance_metadata["num_crewmates"])
        & instance_metadata["observed_impostors"].le(instance_metadata["num_impostors"])
    )
    instance_metadata["instances_for_game"] = instance_metadata.groupby("game_key")["raw_game_instance_key"].transform("size")
    instance_metadata["raw_instance_status"] = "quarantined_incompatible_roster"
    instance_metadata.loc[instance_metadata["roster_is_compatible"] & instance_metadata["instances_for_game"].eq(1), "raw_instance_status"] = "canonical"
    instance_metadata.loc[instance_metadata["instances_for_game"].gt(1), "raw_instance_status"] = "ambiguous_multiple_instances"
    return frame.merge(instance_metadata[["raw_game_instance_key", "observed_crewmates", "observed_impostors", "raw_instance_status"]], on="raw_game_instance_key", how="left", validate="many_to_one")


raw_log_records = load_raw_decision_events()
if raw_log_records.empty:
    raw_decision_events = raw_log_records.copy()
    raw_log_quarantine = raw_log_records.copy()
else:
    raw_decision_events = raw_log_records.loc[raw_log_records["raw_instance_status"].eq("canonical")].copy()
    raw_log_quarantine = raw_log_records.loc[~raw_log_records["raw_instance_status"].eq("canonical")].copy()


## Events dataframe

`events` is a provenance ledger, not an analysis table: it intentionally contains separate raw-decision, discussion-annotation, and final-outcome records, so a spoken utterance can appear more than once across sources. Use the exported `raw_decision_events` table for state/action analyses and `labeled_speech_events` for utterance-level analyses. Raw compact logs add observed player colors, locations, state snapshots, available actions, and selected actions when enabled; they do not claim that an attempted action was confirmed unless `action_accepted` is true.

In [11]:
EVENT_COLUMNS = [
    "event_id",
    "provider",
    "model",
    "condition",
    "num_crewmates",
    "num_impostors",
    "num_players",
    "impostor_ratio",
    "game_id",
    "game_number",
    "game_key",
    "timestep",
    "phase",
    "event_type",
    "actor_id",
    "actor_role",
    "actor_id_source",
    "target_id",
    "target_role",
    "location",
    "available_actions",
    "alive_crewmates",
    "alive_impostors",
    "player_locations",
    "action_text",
    "speech_act",
    "deception_type",
    "death_player",
    "report_player",
    "vote_target",
    "ejected_player",
    "winner_code",
    "winner_side",
    "termination_type",
    "event_source",
    "state_snapshot_available",
    "raw_log_file",
    "raw_game_instance",
    "raw_game_instance_key",
    "observed_crewmates",
    "observed_impostors",
    "raw_instance_status",
    "log_sequence",
    "timestamp",
    "action_source",
    "observed_timestep",
    "max_timesteps_observed",
    "phase_raw",
    "meeting_round",
    "visible_players",
    "visible_player_locations",
    "visible_dead_players",
    "observed_ejection_player",
    "action_accepted",
    "state_inference_source",
    "player_locations_source",
]


def empty_event_columns(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        if column not in frame:
            frame[column] = pd.NA
    return frame[columns]


def normalized_utterance(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()


def target_from_action(action: object) -> object:
    if pd.isna(action):
        return pd.NA
    match = re.search(r"Player\s+\d+\s*:\s*[^,\n]+", str(action), flags=re.IGNORECASE)
    return color_from_player_label(match.group(0)) if match else pd.NA


discussion_keys = discussion[["game_key", "utterance"]].assign(utterance_key=lambda frame: normalized_utterance(frame["utterance"]))
discussion_text_key_counts = discussion_keys.groupby(["game_key", "utterance_key"], dropna=False).size().rename("discussion_records").reset_index()

deception_speaker_candidates = deception[["game_key", "deception_text", "imposter", "deception_type"]].assign(
    utterance_key=lambda frame: normalized_utterance(frame["deception_text"]),
    actor_id=lambda frame: frame["imposter"].astype("string").str.extract(r"^Player\s+\d+\s*:\s*(?P<color>[^:]+)\s*$", expand=True)["color"].str.strip().str.lower(),
)
deception_text_key_counts = deception_speaker_candidates.groupby(["game_key", "utterance_key"], dropna=False).agg(
    deception_records=("actor_id", "size"),
    speaker_ids=("actor_id", "nunique"),
    deception_labels=("deception_type", "nunique"),
).reset_index()

speaker_id_map = (
    deception_speaker_candidates.merge(
        deception_text_key_counts.query("deception_records == 1 and speaker_ids == 1 and deception_labels <= 1"),
        on=["game_key", "utterance_key"],
        how="inner",
    )
    .merge(
        discussion_text_key_counts.query("discussion_records == 1"),
        on=["game_key", "utterance_key"],
        how="inner",
    )
    [["game_key", "utterance_key", "actor_id", "deception_type"]]
    .assign(actor_id_source="deceit_text_exact_match")
)

discussion_events_source = (
    discussion.assign(utterance_key=lambda frame: normalized_utterance(frame["utterance"]))
    .merge(speaker_id_map, on=["game_key", "utterance_key"], how="left", validate="many_to_one")
)
speaker_link_audit = pd.DataFrame(
    {
        "discussion_rows": [len(discussion_events_source)],
        "speaker_ids_recovered": [discussion_events_source["actor_id"].notna().sum()],
        "recovery_rate": [discussion_events_source["actor_id"].notna().mean()],
    }
)
display(speaker_link_audit)

speech_events = discussion_events_source.merge(games[["game_key", "model"]], how="left", on="game_key", validate="many_to_one").assign(
    event_id=lambda frame: "speech|" + frame["game_key"] + "|" + frame["utterance_order"].astype("string"),
    phase="meeting",
    event_type="speech",
    actor_role=lambda frame: frame["player_identity"],
    action_text=lambda frame: frame["utterance"],
    event_source="discussion_annotation",
    state_snapshot_available=False,
)
speech_events = empty_event_columns(speech_events, EVENT_COLUMNS)

outcome_events = games.assign(
    event_id=lambda frame: "game_end|" + frame["game_key"],
    timestep=pd.NA,
    phase="terminal",
    event_type="game_end",
    actor_id=pd.NA,
    actor_id_source=pd.NA,
    actor_role=pd.NA,
    action_text=lambda frame: frame["winner_reason"],
    event_source="summary",
    state_snapshot_available=False,
)
outcome_events = empty_event_columns(outcome_events, EVENT_COLUMNS)

raw_events = raw_decision_events.copy()
if not raw_events.empty:
    raw_events = raw_events.assign(
        event_id=lambda frame: "raw_decision|" + frame["game_key"] + "|" + frame["log_sequence"].astype("string"),
        target_id=lambda frame: frame["action_text"].map(target_from_action),
        target_role=pd.NA,
        player_locations=pd.NA,
        alive_crewmates=pd.NA,
        alive_impostors=pd.NA,
        speech_act=pd.NA,
        deception_type=pd.NA,
        death_player=pd.NA,
        report_player=pd.NA,
        vote_target=pd.NA,
        ejected_player=pd.NA,
        winner_code=pd.NA,
        winner_side=pd.NA,
        termination_type=pd.NA,
        event_source="raw_compact_log",
        state_snapshot_available=True,
        actor_id_source="raw_player_name",
    )
    raw_events = empty_event_columns(raw_events, EVENT_COLUMNS)

event_frames = [speech_events, outcome_events]
if not raw_events.empty:
    event_frames.append(raw_events)
events = pd.concat(event_frames, ignore_index=True)
events["timestep"] = pd.to_numeric(events["timestep"], errors="coerce").astype("Int64")
events = events.sort_values(["provider", "condition", "game_number", "timestep", "event_type"], na_position="last").reset_index(drop=True)


def json_list(value: object) -> list[object]:
    if not isinstance(value, str):
        return []
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return []
    return parsed if isinstance(parsed, list) else []


def json_mapping(value: object) -> dict[str, str]:
    if not isinstance(value, str):
        return {}
    try:
        parsed = json.loads(value)
    except json.JSONDecodeError:
        return {}
    return parsed if isinstance(parsed, dict) else {}


def normalize_action_text(value: object) -> str:
    text = re.sub(r"^\s*(?:\d+\s*[.:]\s*)+", "", str(value))
    return re.sub(r"\s+", " ", text).strip().rstrip(".").casefold()


def action_is_accepted(action: object, offered_actions: object) -> bool:
    if pd.isna(action):
        return False
    normalized_action = normalize_action_text(action)
    available = [normalize_action_text(item) for item in json_list(offered_actions)]
    if normalized_action.startswith("speak:"):
        return any(item.startswith("speak:") for item in available)
    return normalized_action in available


def action_destination(action: object) -> object:
    if pd.isna(action):
        return pd.NA
    match = re.search(r"\bto\s+(?P<location>[^.\n]+?)\s*$", str(action), flags=re.IGNORECASE)
    return match["location"].strip() if match else pd.NA


def reconstruct_raw_state(frame: pd.DataFrame) -> pd.DataFrame:
    raw = frame.loc[frame["event_source"].eq("raw_compact_log")].copy()
    if raw.empty:
        return raw
    raw["event_index"] = raw.index
    raw = raw.sort_values(["game_key", "log_sequence"]).reset_index(drop=True)
    update_columns = [
        "target_role", "alive_crewmates", "alive_impostors", "player_locations",
        "death_player", "report_player", "vote_target", "ejected_player",
        "action_accepted", "state_inference_source", "player_locations_source",
    ]
    updates = {column: [] for column in update_columns}

    for _, game_rows in raw.groupby("game_key", sort=False):
        role_by_color = (
            game_rows.dropna(subset=["actor_id"]).drop_duplicates("actor_id").set_index("actor_id")["actor_role"].to_dict()
        )
        alive_crewmates = int(game_rows["num_crewmates"].iloc[0])
        alive_impostors = int(game_rows["num_impostors"].iloc[0])
        eliminated_players: set[str] = set()
        unknown_player_role = "Crewmate" if game_rows["observed_impostors"].iloc[0] == alive_impostors else pd.NA
        locations: dict[str, str] = {}
        ejections_seen: set[str] = set()
        active_timestep = None
        vote_targets: list[str] = []
        last_vote_update_position = None

        def player_role(player: object) -> object:
            if pd.isna(player):
                return pd.NA
            return role_by_color.get(player, unknown_player_role)

        def remove_player(player: object) -> bool:
            nonlocal alive_crewmates, alive_impostors
            if pd.isna(player) or player in eliminated_players:
                return False
            role = player_role(player)
            if pd.isna(role):
                return False
            if role == "Crewmate":
                alive_crewmates -= 1
            elif role == "Impostor":
                alive_impostors -= 1
            else:
                return False
            eliminated_players.add(player)
            return True

        def finalize_vote_round() -> None:
            nonlocal vote_targets, last_vote_update_position
            if not vote_targets or last_vote_update_position is None:
                vote_targets, last_vote_update_position = [], None
                return
            vote_counts = {target: vote_targets.count(target) for target in set(vote_targets)}
            highest_vote_count = max(vote_counts.values())
            leaders = [target for target, count in vote_counts.items() if count == highest_vote_count]
            if len(leaders) == 1:
                ejected = leaders[0]
                updates["ejected_player"][last_vote_update_position] = ejected
                updates["state_inference_source"][last_vote_update_position] += ";vote_tally"
                ejections_seen.add(ejected)
                remove_player(ejected)
            vote_targets, last_vote_update_position = [], None

        for row in game_rows.itertuples(index=False):
            if active_timestep is not None and row.timestep != active_timestep:
                finalize_vote_round()
            active_timestep = row.timestep
            inferred_from = ["raw_prompt"]
            observed_ejection = row.observed_ejection_player
            ejected_player = pd.NA
            if pd.notna(observed_ejection) and observed_ejection not in ejections_seen:
                ejections_seen.add(observed_ejection)
                remove_player(observed_ejection)
                ejected_player = observed_ejection
                inferred_from.append("meeting_result")

            for dead_player in json_list(row.visible_dead_players):
                if remove_player(dead_player):
                    inferred_from.append("dead_roster")

            if row.phase == "meeting":
                locations.update({player: "Cafeteria" for player in role_by_color})
            locations.update(json_mapping(row.visible_player_locations))
            if pd.notna(row.actor_id) and pd.notna(row.location):
                locations[row.actor_id] = row.location

            accepted = action_is_accepted(row.action_text, row.available_actions)
            target = row.target_id if pd.notna(row.target_id) else pd.NA
            target_role = player_role(target)
            death_player = pd.NA
            report_player = pd.NA
            vote_target = pd.NA

            if accepted and row.event_type == "kill_attempt" and pd.notna(target):
                death_player = target
            elif accepted and row.event_type == "report_attempt":
                report_player = row.actor_id
            elif accepted and row.event_type == "vote_attempt" and pd.notna(target):
                vote_target = target
                vote_targets.append(target)

            updates["target_role"].append(target_role)
            updates["alive_crewmates"].append(alive_crewmates)
            updates["alive_impostors"].append(alive_impostors)
            updates["player_locations"].append(json.dumps(locations, sort_keys=True))
            updates["death_player"].append(death_player)
            updates["report_player"].append(report_player)
            updates["vote_target"].append(vote_target)
            updates["ejected_player"].append(ejected_player)
            updates["action_accepted"].append(accepted)
            updates["state_inference_source"].append(";".join(inferred_from))
            updates["player_locations_source"].append("reconstructed_last_observed")
            if accepted and row.event_type == "vote_attempt" and pd.notna(target):
                last_vote_update_position = len(updates["ejected_player"]) - 1

            if accepted and row.event_type == "kill_attempt" and pd.notna(target):
                remove_player(target)
            if accepted and row.event_type in {"move", "vent_attempt"}:
                destination = action_destination(row.action_text)
                if pd.notna(destination) and pd.notna(row.actor_id):
                    locations[row.actor_id] = destination
        finalize_vote_round()

    for column, values in updates.items():
        raw[column] = values
    for column in update_columns:
        events.loc[raw["event_index"], column] = raw[column].to_numpy()
    return raw


reconstructed_raw_events = reconstruct_raw_state(events)

assert events["event_id"].is_unique, "Event IDs must be unique."
assert set(events["event_type"]).issuperset({"speech", "game_end"}), "Expected annotated speech and outcome events."

print(f"Events: {len(events):,} rows across {events['game_key'].nunique():,} games")
display(events.head(3))

state_fields = [
    "actor_id", "target_id", "location", "available_actions",
    "alive_crewmates", "alive_impostors", "player_locations",
    "death_player", "report_player", "vote_target", "ejected_player",
]
field_availability = pd.DataFrame(
    {"field": state_fields, "observed_rows": [events[field].notna().sum() for field in state_fields]}
)
field_availability["availability"] = field_availability["observed_rows"] / len(events)
display(field_availability)

,discussion_rows,speaker_ids_recovered,recovery_rate
0,120460,18067,0.149983


Events: 527,716 rows across 4,451 games


,event_id,provider,model,condition,num_crewmates,num_impostors,num_players,impostor_ratio,game_id,game_number,game_key,timestep,phase,event_type,actor_id,actor_role,actor_id_source,target_id,target_role,location,available_actions,alive_crewmates,alive_impostors,player_locations,action_text,...,winner_code,winner_side,termination_type,event_source,state_snapshot_available,raw_log_file,raw_game_instance,raw_game_instance_key,observed_crewmates,observed_impostors,raw_instance_status,log_sequence,timestamp,action_source,observed_timestep,max_timesteps_observed,phase_raw,meeting_round,visible_players,visible_player_locations,visible_dead_players,observed_ejection_player,action_accepted,state_inference_source,player_locations_source
0,raw_decision|CLAUDE|3V1|1|20,CLAUDE,claude-sonnet-4-6,3V1,3,1,4,0.25,Game 1,1,CLAUDE|3V1|1,0,task,move,black,Crewmate,raw_player_name,NaN,NaN,Cafeteria,"[""MOVE from Cafeteria to Admin"", ""MOVE from Ca...",3,1,"{""black"": ""Cafeteria"", ""blue"": ""Cafeteria"", ""o...",MOVE from Cafeteria to Admin,...,<NA>,<NA>,<NA>,raw_compact_log,True,data/logs/CLAUDE_LOGS/CLAUDE_3V1/agent-logs-co...,1,CLAUDE|3V1|1|raw_instance_1,3,1,canonical,20,2026-06-11 20:01:47.600552,response.thinking_process.action,0,20,Task phase,<NA>,"[""Player 1: black"", ""Player 2: blue"", ""Player ...","{""black"": ""Cafeteria"", ""blue"": ""Cafeteria"", ""w...",[],<NA>,True,raw_prompt,reconstructed_last_observed
1,raw_decision|CLAUDE|3V1|1|90,CLAUDE,claude-sonnet-4-6,3V1,3,1,4,0.25,Game 1,1,CLAUDE|3V1|1,0,task,move,blue,Crewmate,raw_player_name,NaN,NaN,Cafeteria,"[""MOVE from Cafeteria to Admin"", ""MOVE from Ca...",3,1,"{""black"": ""Admin"", ""blue"": ""Cafeteria"", ""orang...",MOVE from Cafeteria to Admin,...,<NA>,<NA>,<NA>,raw_compact_log,True,data/logs/CLAUDE_LOGS/CLAUDE_3V1/agent-logs-co...,1,CLAUDE|3V1|1|raw_instance_1,3,1,canonical,90,2026-06-11 20:01:56.240081,response.thinking_process.action,0,20,Task phase,<NA>,"[""Player 2: blue"", ""Player 3: white"", ""Player ...","{""blue"": ""Cafeteria"", ""white"": ""Cafeteria"", ""o...",[],<NA>,True,raw_prompt,reconstructed_last_observed
2,raw_decision|CLAUDE|3V1|1|141,CLAUDE,claude-sonnet-4-6,3V1,3,1,4,0.25,Game 1,1,CLAUDE|3V1|1,0,task,move,white,Crewmate,raw_player_name,NaN,NaN,Cafeteria,"[""MOVE from Cafeteria to Admin"", ""MOVE from Ca...",3,1,"{""black"": ""Admin"", ""blue"": ""Admin"", ""orange"": ...",MOVE from Cafeteria to Admin,...,<NA>,<NA>,<NA>,raw_compact_log,True,data/logs/CLAUDE_LOGS/CLAUDE_3V1/agent-logs-co...,1,CLAUDE|3V1|1|raw_instance_1,3,1,canonical,141,2026-06-11 20:02:05.038476,response.thinking_process.action,0,20,Task phase,<NA>,"[""Player 3: white"", ""Player 4: orange""]","{""white"": ""Cafeteria"", ""orange"": ""Cafeteria""}",[],<NA>,True,raw_prompt,reconstructed_last_observed


,field,observed_rows,availability
0,actor_id,420923,0.797632
1,target_id,47492,0.089995
2,location,402856,0.763395
3,available_actions,402856,0.763395
4,alive_crewmates,402856,0.763395
5,alive_impostors,402856,0.763395
6,player_locations,402856,0.763395
7,death_player,7754,0.014694
8,report_player,2736,0.005185
9,vote_target,36089,0.068387


## Export analysis-ready datasets

This export is intentionally available only after a complete raw-log run. It preserves the raw decision ledger separately, then creates a narrow labeled-speech table by conservatively matching accepted raw `SPEAK` decisions to the speech-act and deception annotations. The audit files retain every unmatched or ambiguous join category.


In [15]:
if RAW_LOG_MODE != "all":
    raise RuntimeError("Set RAW_LOG_MODE = 'all' before exporting derived datasets.")

DERIVED_DIR = PROJECT_ROOT / "derived" / "analysis"
DERIVED_DIR.mkdir(parents=True, exist_ok=True)


def normalized_speech_text(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.replace(r"^\s*SPEAK\s*:?\s*", "", regex=True, case=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.replace(r"^[\"']+|[\"']+$", "", regex=True)
        .str.casefold()
    )


SPEECH_JOIN_KEYS = ["provider", "condition", "game_number", "timestep", "utterance_key"]
MIN_SPEECH_MATCH_RATE = 0.99
print("Speech matching version: outer_quote_normalization_v1")

raw_speech = (
    events.loc[
        events["event_source"].eq("raw_compact_log")
        & events["event_type"].eq("speech")
        & events["action_accepted"].fillna(False),
    ]
    .copy()
    .assign(utterance=lambda frame: frame["action_text"].str.replace(r"^\s*SPEAK\s*:?\s*", "", regex=True, case=False))
)
raw_speech["utterance_key"] = normalized_speech_text(raw_speech["utterance"])

discussion_for_join = discussion.copy()
discussion_for_join["utterance_key"] = normalized_speech_text(discussion_for_join["utterance"])

raw_key_counts = raw_speech.groupby(SPEECH_JOIN_KEYS, dropna=False).size().rename("raw_records").reset_index()
discussion_match_key_counts = discussion_for_join.groupby(SPEECH_JOIN_KEYS, dropna=False).size().rename("discussion_records").reset_index()
speech_key_audit = raw_key_counts.merge(discussion_match_key_counts, how="outer", on=SPEECH_JOIN_KEYS).fillna(0)
speech_key_audit[["raw_records", "discussion_records"]] = speech_key_audit[["raw_records", "discussion_records"]].astype(int)
speech_key_audit["join_status"] = "ambiguous"
speech_key_audit.loc[(speech_key_audit["raw_records"] == 1) & (speech_key_audit["discussion_records"] == 1), "join_status"] = "matched_one_to_one"
speech_key_audit.loc[(speech_key_audit["raw_records"] > 0) & (speech_key_audit["discussion_records"] == 0), "join_status"] = "raw_without_discussion"
speech_key_audit.loc[(speech_key_audit["raw_records"] == 0) & (speech_key_audit["discussion_records"] > 0), "join_status"] = "discussion_without_raw"

raw_speech = raw_speech.merge(speech_key_audit[SPEECH_JOIN_KEYS + ["join_status", "discussion_records"]], how="left", on=SPEECH_JOIN_KEYS, validate="many_to_one")
discussion_for_join = discussion_for_join.merge(speech_key_audit[SPEECH_JOIN_KEYS + ["join_status", "raw_records"]], how="left", on=SPEECH_JOIN_KEYS, validate="many_to_one")
raw_speech["join_status"] = raw_speech["join_status"].fillna("missing_join_key")
discussion_for_join["join_status"] = discussion_for_join["join_status"].fillna("missing_join_key")

matched_raw_speech = raw_speech.loc[raw_speech["join_status"].eq("matched_one_to_one")].drop(columns=["speech_act", "deception_type"], errors="ignore").copy()
matched_discussion = discussion_for_join.loc[discussion_for_join["join_status"].eq("matched_one_to_one")].copy()
labeled_speech_events = matched_raw_speech.merge(
    matched_discussion[SPEECH_JOIN_KEYS + ["player_identity", "speech_act", "speech_act_raw", "label_status"]],
    how="inner",
    on=SPEECH_JOIN_KEYS,
    validate="one_to_one",
    suffixes=("", "_discussion"),
)
labeled_speech_events["role_matches_discussion"] = labeled_speech_events["actor_role"].eq(labeled_speech_events["player_identity"])

# Reclassify placeholders and safely resolve duplicate speech only when its labels agree.
PLACEHOLDER_SPEECH_KEYS = frozenset({"", "...", "speak", "speak:", "none", "n/a", "na"})


def role_signature(frame: pd.DataFrame, role_column: str) -> pd.DataFrame:
    return (
        frame.groupby(SPEECH_JOIN_KEYS, dropna=False)[role_column]
        .apply(lambda roles: tuple(sorted(roles.astype("string").fillna("<missing>").value_counts().items())))
        .rename("role_signature")
        .reset_index()
    )


for frame in [raw_speech, discussion_for_join]:
    frame["speech_content_status"] = "contentful"
    frame.loc[frame["utterance_key"].isin(PLACEHOLDER_SPEECH_KEYS), "speech_content_status"] = "non_content_placeholder"

contentful_raw_speech = raw_speech.loc[raw_speech["speech_content_status"].eq("contentful")].copy()
contentful_discussion = discussion_for_join.loc[discussion_for_join["speech_content_status"].eq("contentful")].copy()
raw_key_counts = contentful_raw_speech.groupby(SPEECH_JOIN_KEYS, dropna=False).size().rename("raw_records").reset_index()
discussion_match_key_counts = contentful_discussion.groupby(SPEECH_JOIN_KEYS, dropna=False).size().rename("discussion_records").reset_index()
raw_role_signatures = role_signature(contentful_raw_speech, "actor_role").rename(columns={"role_signature": "raw_role_signature"})
discussion_role_signatures = role_signature(contentful_discussion, "player_identity").rename(columns={"role_signature": "discussion_role_signature"})
discussion_label_audit = (
    contentful_discussion.groupby(SPEECH_JOIN_KEYS, dropna=False)
    .agg(discussion_speech_acts=("speech_act", lambda labels: labels.nunique(dropna=False)))
    .reset_index()
)
speech_key_audit = (
    raw_key_counts.merge(discussion_match_key_counts, how="outer", on=SPEECH_JOIN_KEYS)
    .merge(raw_role_signatures, how="left", on=SPEECH_JOIN_KEYS)
    .merge(discussion_role_signatures, how="left", on=SPEECH_JOIN_KEYS)
    .merge(discussion_label_audit, how="left", on=SPEECH_JOIN_KEYS)
    .fillna({"raw_records": 0, "discussion_records": 0})
)
speech_key_audit[["raw_records", "discussion_records"]] = speech_key_audit[["raw_records", "discussion_records"]].astype(int)
speech_key_audit["join_status"] = "ambiguous"
speech_key_audit.loc[(speech_key_audit["raw_records"] == 1) & (speech_key_audit["discussion_records"] == 1), "join_status"] = "matched_one_to_one"
equivalent_duplicates = (
    speech_key_audit["raw_records"].eq(speech_key_audit["discussion_records"])
    & speech_key_audit["raw_records"].gt(1)
    & speech_key_audit["raw_role_signature"].eq(speech_key_audit["discussion_role_signature"])
    & speech_key_audit["discussion_speech_acts"].eq(1)
)
speech_key_audit.loc[equivalent_duplicates, "join_status"] = "matched_duplicate_equivalent"
duplicate_label_conflicts = (
    speech_key_audit["raw_records"].eq(speech_key_audit["discussion_records"])
    & speech_key_audit["raw_records"].gt(1)
    & speech_key_audit["raw_role_signature"].eq(speech_key_audit["discussion_role_signature"])
    & ~speech_key_audit["discussion_speech_acts"].eq(1)
)
speech_key_audit.loc[duplicate_label_conflicts, "join_status"] = "matched_duplicate_label_conflict"
annotation_count_mismatches = (
    speech_key_audit["raw_records"].gt(0)
    & speech_key_audit["discussion_records"].gt(0)
    & speech_key_audit["join_status"].eq("ambiguous")
)
speech_key_audit.loc[annotation_count_mismatches, "join_status"] = "matched_annotation_count_mismatch"
speech_key_audit.loc[(speech_key_audit["raw_records"] > 0) & (speech_key_audit["discussion_records"] == 0), "join_status"] = "raw_without_discussion"
speech_key_audit.loc[(speech_key_audit["raw_records"] == 0) & (speech_key_audit["discussion_records"] > 0), "join_status"] = "discussion_without_raw"

raw_speech = raw_speech.drop(columns=["join_status", "discussion_records"], errors="ignore").merge(
    speech_key_audit[SPEECH_JOIN_KEYS + ["join_status", "discussion_records"]], how="left", on=SPEECH_JOIN_KEYS, validate="many_to_one"
)
discussion_for_join = discussion_for_join.drop(columns=["join_status", "raw_records"], errors="ignore").merge(
    speech_key_audit[SPEECH_JOIN_KEYS + ["join_status", "raw_records"]], how="left", on=SPEECH_JOIN_KEYS, validate="many_to_one"
)
raw_speech["join_status"] = raw_speech["join_status"].fillna(raw_speech["speech_content_status"])
discussion_for_join["join_status"] = discussion_for_join["join_status"].fillna(discussion_for_join["speech_content_status"])

matched_raw_speech = raw_speech.loc[raw_speech["join_status"].eq("matched_one_to_one")].drop(columns=["speech_act", "deception_type"], errors="ignore").copy()
matched_discussion = discussion_for_join.loc[discussion_for_join["join_status"].eq("matched_one_to_one")].copy()
one_to_one_labeled_speech = matched_raw_speech.merge(
    matched_discussion[SPEECH_JOIN_KEYS + ["player_identity", "speech_act", "speech_act_raw", "label_status"]],
    how="inner", on=SPEECH_JOIN_KEYS, validate="one_to_one", suffixes=("", "_discussion"),
)
one_to_one_labeled_speech["role_matches_discussion"] = one_to_one_labeled_speech["actor_role"].eq(one_to_one_labeled_speech["player_identity"])
one_to_one_labeled_speech["speech_join_method"] = "one_to_one"
duplicate_raw_speech = raw_speech.loc[raw_speech["join_status"].eq("matched_duplicate_equivalent")].drop(columns=["speech_act", "deception_type"], errors="ignore").copy()
duplicate_discussion_labels = (
    discussion_for_join.loc[discussion_for_join["join_status"].eq("matched_duplicate_equivalent")]
    .groupby(SPEECH_JOIN_KEYS, as_index=False)
    .agg(speech_act=("speech_act", "first"), speech_act_raw=("speech_act_raw", "first"), label_status=("label_status", "first"))
)
duplicate_labeled_speech = duplicate_raw_speech.merge(duplicate_discussion_labels, how="inner", on=SPEECH_JOIN_KEYS, validate="many_to_one")
duplicate_labeled_speech["player_identity"] = duplicate_labeled_speech["actor_role"]
duplicate_labeled_speech["role_matches_discussion"] = True
duplicate_labeled_speech["speech_join_method"] = "duplicate_equivalent_label"
labeled_speech_events = pd.concat([one_to_one_labeled_speech, duplicate_labeled_speech], ignore_index=True)

deception_for_join = deception[["game_key", "imposter", "deception_text", "deception_type", "deception_raw", "label_status"]].copy()
deception_for_join = deception_for_join.assign(
    actor_id=lambda frame: frame["imposter"].map(color_from_player_label),
    utterance_key=lambda frame: normalized_speech_text(frame["deception_text"]),
)
DECEPTION_JOIN_KEYS = ["game_key", "actor_id", "utterance_key"]
deception_match_key_counts = deception_for_join.groupby(DECEPTION_JOIN_KEYS, dropna=False).size().rename("deception_records").reset_index()
labeled_speech_events = labeled_speech_events.merge(deception_match_key_counts, how="left", on=DECEPTION_JOIN_KEYS, validate="many_to_one")
labeled_speech_events["deception_join_status"] = "not_applicable_crewmate"
impostor_speech = labeled_speech_events["actor_role"].eq("Impostor")
labeled_speech_events.loc[impostor_speech, "deception_join_status"] = "impostor_without_deception_record"
labeled_speech_events.loc[impostor_speech & labeled_speech_events["deception_records"].eq(1), "deception_join_status"] = "matched_one_to_one"
labeled_speech_events.loc[impostor_speech & labeled_speech_events["deception_records"].gt(1), "deception_join_status"] = "ambiguous"

labeled_speech_events = labeled_speech_events.drop(columns=["winner_code", "winner_side", "termination_type"], errors="ignore")
labeled_speech_events = labeled_speech_events.merge(
    deception_for_join.loc[deception_for_join.duplicated(DECEPTION_JOIN_KEYS, keep=False) == False, DECEPTION_JOIN_KEYS + ["deception_type", "deception_raw", "label_status"]],
    how="left",
    on=DECEPTION_JOIN_KEYS,
    validate="many_to_one",
    suffixes=("", "_deception"),
)
labeled_speech_events = labeled_speech_events.merge(
    games[["game_key", "winner_code", "winner_reason", "winner_side", "termination_type", "impostor_win"]],
    how="left",
    on="game_key",
    validate="many_to_one",
)

raw_speech_audit = raw_speech.groupby(["provider", "condition", "join_status"], dropna=False).size().rename("records").reset_index().assign(audit_type="raw_to_discussion")
discussion_speech_audit = discussion_for_join.groupby(["provider", "condition", "join_status"], dropna=False).size().rename("records").reset_index().assign(audit_type="discussion_to_raw")
MATCHED_SPEECH_STATUSES = {"matched_one_to_one", "matched_duplicate_equivalent", "matched_duplicate_label_conflict", "matched_annotation_count_mismatch"}
provider_speech_coverage = (
    raw_speech.loc[raw_speech["speech_content_status"].eq("contentful")]
    .assign(matched_contentful=lambda frame: frame["join_status"].isin(MATCHED_SPEECH_STATUSES))
    .groupby("provider")
    .agg(contentful_raw_speech_records=("event_id", "size"), matched_contentful_records=("matched_contentful", "sum"))
    .reset_index()
)
provider_speech_coverage["match_rate"] = provider_speech_coverage["matched_contentful_records"] / provider_speech_coverage["contentful_raw_speech_records"]
if not provider_speech_coverage["match_rate"].ge(MIN_SPEECH_MATCH_RATE).all():
    speech_match_failure_audit = (
        raw_speech.loc[raw_speech["speech_content_status"].eq("contentful")]
        .groupby(["provider", "join_status"], dropna=False)
        .size().rename("records").reset_index()
    )
    display(speech_match_failure_audit)
    llama_match_examples = raw_speech.loc[
        raw_speech["provider"].eq("LLAMA")
        & raw_speech["join_status"].isin({"ambiguous", "raw_without_discussion"}),
        ["join_status", "condition", "game_number", "timestep", "actor_role", "utterance", "discussion_records"],
    ].drop_duplicates().sort_values(["join_status", "condition", "game_number", "timestep"])
    display(llama_match_examples.head(20))
assert provider_speech_coverage["match_rate"].ge(MIN_SPEECH_MATCH_RATE).all(), (
    "Raw-to-discussion speech matching fell below the required threshold: "
    + provider_speech_coverage.to_string(index=False)
)
deception_audit = labeled_speech_events.groupby(["provider", "condition", "deception_join_status"], dropna=False).size().rename("records").reset_index().rename(columns={"deception_join_status": "join_status"}).assign(audit_type="labeled_speech_to_deception")
role_audit = labeled_speech_events.groupby(["provider", "condition", "role_matches_discussion"], dropna=False).size().rename("records").reset_index().assign(join_status=lambda frame: frame["role_matches_discussion"].map({True: "role_match", False: "role_mismatch"}), audit_type="raw_role_to_discussion_role")[["provider", "condition", "join_status", "records", "audit_type"]]
join_audit = pd.concat([raw_speech_audit, discussion_speech_audit, deception_audit, role_audit], ignore_index=True)[["audit_type", "provider", "condition", "join_status", "records"]].sort_values(["audit_type", "provider", "condition", "join_status"]).reset_index(drop=True)

raw_join_records = raw_speech[["event_id", "provider", "condition", "game_key", "game_number", "timestep", "actor_id", "actor_role", "utterance", "utterance_key", "join_status", "discussion_records"]].rename(columns={"event_id": "source_record_id", "discussion_records": "candidate_records"}).assign(audit_source="raw_speech")
discussion_join_records = discussion_for_join[["utterance_order", "provider", "condition", "game_key", "game_number", "timestep", "player_identity", "utterance", "utterance_key", "join_status", "raw_records"]].rename(columns={"utterance_order": "source_record_id", "player_identity": "actor_role", "raw_records": "candidate_records"}).assign(audit_source="discussion_annotation", actor_id=pd.NA)
join_record_audit = pd.concat([raw_join_records, discussion_join_records], ignore_index=True)[["audit_source", "source_record_id", "provider", "condition", "game_key", "game_number", "timestep", "actor_id", "actor_role", "utterance", "utterance_key", "join_status", "candidate_records"]]
join_record_audit["source_record_id"] = join_record_audit["source_record_id"].astype("string")

DERIVED_LOCATION_RENAMES = {
    "player_locations": "last_observed_player_locations",
    "player_locations_source": "last_observed_player_locations_source",
}
raw_decision_export = events.loc[events["event_source"].eq("raw_compact_log")].rename(columns=DERIVED_LOCATION_RENAMES).copy()
raw_log_quarantine_export = raw_log_quarantine.rename(columns=DERIVED_LOCATION_RENAMES).copy()
raw_instance_audit = (
    raw_log_records.groupby(["provider", "condition", "game_key", "raw_game_instance_key", "raw_instance_status"], dropna=False)
    .agg(decision_records=("log_sequence", "size"), observed_crewmates=("observed_crewmates", "first"), observed_impostors=("observed_impostors", "first"))
    .reset_index()
)
canonical_game_keys = set(raw_decision_export["game_key"].dropna())
game_coverage_audit = games[["provider", "condition", "game_key"]].copy()
game_coverage_audit["has_canonical_raw_log"] = game_coverage_audit["game_key"].isin(canonical_game_keys)
game_coverage_audit["coverage_status"] = game_coverage_audit["has_canonical_raw_log"].map({True: "canonical_raw_log", False: "missing_or_ambiguous_raw_log"})
labeled_speech_events = labeled_speech_events.rename(columns=DERIVED_LOCATION_RENAMES)
raw_decision_export.to_parquet(DERIVED_DIR / "raw_decision_events.parquet", index=False)
raw_log_quarantine_export.to_parquet(DERIVED_DIR / "raw_log_quarantine.parquet", index=False)
labeled_speech_events.to_parquet(DERIVED_DIR / "labeled_speech_events.parquet", index=False)
games.to_parquet(DERIVED_DIR / "games.parquet", index=False)
join_audit.to_parquet(DERIVED_DIR / "join_audit.parquet", index=False)
join_record_audit.to_parquet(DERIVED_DIR / "speech_join_audit.parquet", index=False)
raw_instance_audit.to_parquet(DERIVED_DIR / "raw_instance_audit.parquet", index=False)
game_coverage_audit.to_parquet(DERIVED_DIR / "game_coverage_audit.parquet", index=False)

export_manifest = {
    "raw_decision_events": len(raw_decision_export),
    "discussion_annotations": len(discussion),
    "deception_annotations": len(deception),
    "labeled_speech_events": len(labeled_speech_events),
    "speech_joined_one_to_one": int(raw_speech["join_status"].eq("matched_one_to_one").sum()),
    "speech_joined_duplicate_equivalent": int(raw_speech["join_status"].eq("matched_duplicate_equivalent").sum()),
    "speech_matched_duplicate_label_conflict": int(raw_speech["join_status"].eq("matched_duplicate_label_conflict").sum()),
    "speech_matched_annotation_count_mismatch": int(raw_speech["join_status"].eq("matched_annotation_count_mismatch").sum()),
    "speech_non_content_placeholders": int(raw_speech["join_status"].eq("non_content_placeholder").sum()),
    "speech_ambiguous": int(raw_speech["join_status"].eq("ambiguous").sum()),
    "speech_without_discussion": int(raw_speech["join_status"].eq("raw_without_discussion").sum()),
    "impostor_deception_joined_one_to_one": int(labeled_speech_events["deception_join_status"].eq("matched_one_to_one").sum()),
    "speech_match_rate_by_provider": {row.provider: float(row.match_rate) for row in provider_speech_coverage.itertuples(index=False)},
    "outputs": {name: str((DERIVED_DIR / name).relative_to(PROJECT_ROOT)) for name in ["raw_decision_events.parquet", "raw_log_quarantine.parquet", "labeled_speech_events.parquet", "games.parquet", "join_audit.parquet", "speech_join_audit.parquet", "raw_instance_audit.parquet", "game_coverage_audit.parquet", "export_manifest.json"]},
}
(DERIVED_DIR / "export_manifest.json").write_text(json.dumps(export_manifest, indent=2) + "\n", encoding="utf-8")

display(join_audit)
display(pd.DataFrame([export_manifest]))


Speech matching version: outer_quote_normalization_v1


,audit_type,provider,condition,join_status,records
0,discussion_to_raw,CLAUDE,3V1,matched_one_to_one,516
1,discussion_to_raw,CLAUDE,4V1,discussion_without_raw,2
2,discussion_to_raw,CLAUDE,4V1,matched_one_to_one,955
3,discussion_to_raw,CLAUDE,4V2,matched_one_to_one,759
4,discussion_to_raw,CLAUDE,5V1,discussion_without_raw,3
...,...,...,...,...,...
464,raw_to_discussion,OPENAI,6V1,matched_one_to_one,2897
465,raw_to_discussion,OPENAI,6V2,matched_one_to_one,4151
466,raw_to_discussion,OPENAI,6V3,matched_one_to_one,3232
467,raw_to_discussion,OPENAI,7V1,matched_one_to_one,3948


,raw_decision_events,discussion_annotations,deception_annotations,labeled_speech_events,speech_joined_one_to_one,speech_joined_duplicate_equivalent,speech_matched_duplicate_label_conflict,speech_matched_annotation_count_mismatch,speech_non_content_placeholders,speech_ambiguous,speech_without_discussion,impostor_deception_joined_one_to_one,speech_match_rate_by_provider,outputs
0,402856,120460,27297,109352,105468,3884,141,144,1902,0,490,25484,"{'CLAUDE': 0.998530029399412, 'GEMINI': 0.9999...",{'raw_decision_events.parquet': 'derived/analy...


In [ ]:
analysis_tables = {
    "raw_decision_events": raw_decision_export,
    "raw_log_quarantine": raw_log_quarantine_export,
    "labeled_speech_events": labeled_speech_events,
    "games": games,
    "join_audit": join_audit,
    "speech_join_audit": join_record_audit,
    "raw_instance_audit": raw_instance_audit,
    "game_coverage_audit": game_coverage_audit,
}

assert labeled_speech_events["event_id"].is_unique
assert labeled_speech_events["winner_side"].notna().all()
assert raw_decision_export["event_source"].eq("raw_compact_log").all()
assert raw_decision_export["raw_instance_status"].eq("canonical").all()
assert set(join_audit["audit_type"]) == {
    "raw_to_discussion",
    "discussion_to_raw",
    "labeled_speech_to_deception",
    "raw_role_to_discussion_role",
}
